In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations
from pathlib import Path
from datetime import datetime, timedelta
import earthaccess
from shapely.geometry import box

# =========================
# USER CONFIG
# =========================
START_DATE = "2020-01-01"
END_DATE   = "2025-12-31"

# Example Landsat info
DATE_ACQUIRED = "2020-01-13"
SCENE_CENTER_TIME = "01:46:54.3099020Z"
STATION_ID = "LGN"

# =========================
# NEW AOI: 
# lon, lat in EPSG:4326
# =========================
BUFFER_DEG = 0.05

points = [
    (125.2551, -8.3528),
    (125.7495, -8.3528),
    (125.1672, -9.1618),
    (125.7056, -9.2160),
]

lons = [p[0] for p in points]
lats = [p[1] for p in points]

min_lon, max_lon = min(lons), max(lons)
min_lat, max_lat = min(lats), max(lats)

width = max_lon - min_lon
height = max_lat - min_lat
side = max(width, height) + 2 * BUFFER_DEG

center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2
half_side = side / 2

square_min_lon = center_lon - half_side
square_max_lon = center_lon + half_side
square_min_lat = center_lat - half_side
square_max_lat = center_lat + half_side

square_geom_4326 = box(
    square_min_lon,
    square_min_lat,
    square_max_lon,
    square_max_lat,
)

bbox_search = square_geom_4326.bounds  # west, south, east, north

print("AOI bounds EPSG:4326:", bbox_search)

# =========================
# VIIRS CONFIG
# =========================
SHORT_NAME_PM = "L3S_LEO_PM-STAR-v2.81"
USE_PM = True

# OUT_ROOT = (
#     Path("/Volumes/purkislab2a/")
#     / "Mingyue"
#     / "West_Fl_Shelf"
#     / "L3S_STAR"
# )
OUT_ROOT = Path("../Timor_part1/L3S_STAR")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# HELPERS
# =========================
def month_start(dt: datetime) -> datetime:
    return dt.replace(day=1)

def month_end(dt: datetime) -> datetime:
    if dt.month == 12:
        nxt = dt.replace(year=dt.year + 1, month=1, day=1)
    else:
        nxt = dt.replace(month=dt.month + 1, day=1)
    return nxt - timedelta(days=1)

def iter_months(start: datetime, end: datetime):
    cur = month_start(start)
    while cur <= end:
        me = month_end(cur)
        yield cur, min(me, end)

        if cur.month == 12:
            cur = cur.replace(year=cur.year + 1, month=1, day=1)
        else:
            cur = cur.replace(month=cur.month + 1, day=1)

def download_one_month(
    short_name: str,
    t0: datetime,
    t1: datetime,
    bbox4,
    out_dir: Path,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    results = earthaccess.search_data(
        short_name=short_name,
        provider="POCLOUD",
        temporal=(
            t0.strftime("%Y-%m-%dT00:00:00Z"),
            t1.strftime("%Y-%m-%dT23:59:59Z"),
        ),
        bounding_box=bbox4,
    )

    print(
        f"[SEARCH] {short_name} "
        f"{t0:%Y-%m-%d}..{t1:%Y-%m-%d} -> {len(results)} granules"
    )

    if len(results) == 0:
        return []

    paths = earthaccess.download(results, local_path=str(out_dir))
    paths = paths or []

    print(f"[DL] {short_name} -> {len(paths)} files")
    return paths

# =========================
# MAIN
# =========================
def main():
    auth = earthaccess.login(strategy="interactive", persist=True)
    print("Authenticated:", auth.authenticated)

    start = datetime.fromisoformat(START_DATE)
    end = datetime.fromisoformat(END_DATE)

    for m0, m1 in iter_months(start, end):
        month_tag = m0.strftime("%Y-%m")
        print(f"\n=== Download month {month_tag} ===")

        month_dir = OUT_ROOT / "raw" / month_tag / "PM"
        month_dir.mkdir(parents=True, exist_ok=True)

        total = 0

        if USE_PM:
            total += len(
                download_one_month(
                    SHORT_NAME_PM,
                    m0,
                    m1,
                    bbox_search,
                    month_dir,
                )
            )

        if total == 0:
            print("No files this month.")

    print("\nDONE. Raw netCDF saved under:", OUT_ROOT / "raw")

if __name__ == "__main__":
    main()

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta
import earthaccess
from shapely.geometry import box

# =========================
# USER CONFIG
# =========================
START_DATE = "2020-01-01"
END_DATE   = "2025-12-31"

# Landsat scene UTC center time
# Used only to decide preferred VIIRS AM/PM search order
SCENE_CENTER_TIME = "01:46:54.3099020Z"

# =========================
# AOI: lon, lat in EPSG:4326
# =========================
BUFFER_DEG = 0.05

points = [
    (125.2551, -8.3528),
    (125.7495, -8.3528),
    (125.1672, -9.1618),
    (125.7056, -9.2160),
]

lons = [p[0] for p in points]
lats = [p[1] for p in points]

min_lon, max_lon = min(lons), max(lons)
min_lat, max_lat = min(lats), max(lats)

width = max_lon - min_lon
height = max_lat - min_lat
side = max(width, height) + 2 * BUFFER_DEG

center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2
half_side = side / 2

square_min_lon = center_lon - half_side
square_max_lon = center_lon + half_side
square_min_lat = center_lat - half_side
square_max_lat = center_lat + half_side

square_geom_4326 = box(
    square_min_lon,
    square_min_lat,
    square_max_lon,
    square_max_lat,
)

bbox_search = square_geom_4326.bounds  # west, south, east, north

print("AOI bounds EPSG:4326:", bbox_search)
print("AOI center:", center_lon, center_lat)

# =========================
# VIIRS CONFIG
# =========================
SHORT_NAME_AM = "L3S_LEO_AM-STAR-v2.81"
SHORT_NAME_PM = "L3S_LEO_PM-STAR-v2.81"

OUT_ROOT = Path("../Timor_part1/L3S_STAR")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# AM / PM PREFERENCE
# =========================
def choose_preferred_viirs_pass(scene_center_time: str, center_lon: float):
    """
    Decide preferred VIIRS pass using Landsat scene center time.

    SCENE_CENTER_TIME is UTC.
    Approx local solar time = UTC hour + lon / 15.

    local solar time < 12  -> prefer AM first
    local solar time >= 12 -> prefer PM first

    If preferred pass has no granules, script will try the other pass.
    """
    time_str = scene_center_time.replace("Z", "")

    hh, mm, ss = time_str.split(":")
    utc_hour = int(hh) + int(mm) / 60.0 + float(ss) / 3600.0

    local_solar_hour = (utc_hour + center_lon / 15.0) % 24.0

    if local_solar_hour < 12:
        preferred = "AM"
        fallback = "PM"
    else:
        preferred = "PM"
        fallback = "AM"

    return preferred, fallback, utc_hour, local_solar_hour


PREFERRED_PASS, FALLBACK_PASS, UTC_HOUR, LOCAL_SOLAR_HOUR = choose_preferred_viirs_pass(
    SCENE_CENTER_TIME,
    center_lon,
)

SHORT_NAME_LOOKUP = {
    "AM": SHORT_NAME_AM,
    "PM": SHORT_NAME_PM,
}

SEARCH_ORDER = [PREFERRED_PASS, FALLBACK_PASS]

print("\n=== VIIRS PASS SEARCH ORDER ===")
print("SCENE_CENTER_TIME UTC:", SCENE_CENTER_TIME)
print("UTC hour:", round(UTC_HOUR, 3))
print("Approx local solar hour:", round(LOCAL_SOLAR_HOUR, 3))
print("Search order:", SEARCH_ORDER)
print("Preferred short name:", SHORT_NAME_LOOKUP[PREFERRED_PASS])
print("Fallback short name:", SHORT_NAME_LOOKUP[FALLBACK_PASS])

# =========================
# HELPERS
# =========================
def month_start(dt: datetime) -> datetime:
    return dt.replace(day=1)


def month_end(dt: datetime) -> datetime:
    if dt.month == 12:
        nxt = dt.replace(year=dt.year + 1, month=1, day=1)
    else:
        nxt = dt.replace(month=dt.month + 1, day=1)

    return nxt - timedelta(days=1)


def iter_months(start: datetime, end: datetime):
    cur = month_start(start)

    while cur <= end:
        me = month_end(cur)
        yield cur, min(me, end)

        if cur.month == 12:
            cur = cur.replace(year=cur.year + 1, month=1, day=1)
        else:
            cur = cur.replace(month=cur.month + 1, day=1)


def search_one_month(
    short_name: str,
    t0: datetime,
    t1: datetime,
    bbox4,
):
    results = earthaccess.search_data(
        short_name=short_name,
        provider="POCLOUD",
        temporal=(
            t0.strftime("%Y-%m-%dT00:00:00Z"),
            t1.strftime("%Y-%m-%dT23:59:59Z"),
        ),
        bounding_box=bbox4,
    )

    print(
        f"[SEARCH] {short_name} "
        f"{t0:%Y-%m-%d}..{t1:%Y-%m-%d} -> {len(results)} granules"
    )

    return results


def download_results(
    results,
    out_dir: Path,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(results) == 0:
        return []

    paths = earthaccess.download(
        results,
        local_path=str(out_dir),
    )

    paths = paths or []

    print(f"[DL] -> {len(paths)} files")
    return paths


# =========================
# MAIN
# =========================
def main():
    auth = earthaccess.login(
        strategy="interactive",
        persist=True,
    )

    print("Authenticated:", auth.authenticated)

    start = datetime.fromisoformat(START_DATE)
    end = datetime.fromisoformat(END_DATE)

    print("\n=== DOWNLOAD SETTINGS ===")
    print("Date range:", START_DATE, "to", END_DATE)
    print("Output root:", OUT_ROOT)
    print("Search order:", SEARCH_ORDER)

    for m0, m1 in iter_months(start, end):
        month_tag = m0.strftime("%Y-%m")

        print(f"\n=== Download month {month_tag} ===")

        downloaded_this_month = 0
        selected_pass = None

        for pass_name in SEARCH_ORDER:
            short_name = SHORT_NAME_LOOKUP[pass_name]

            results = search_one_month(
                short_name,
                m0,
                m1,
                bbox_search,
            )

            if len(results) == 0:
                print(f"No {pass_name} files this month. Trying next pass.")
                continue

            month_dir = OUT_ROOT / "raw" / month_tag / pass_name

            paths = download_results(
                results,
                month_dir,
            )

            downloaded_this_month += len(paths)
            selected_pass = pass_name

            # stop after first available pass
            break

        if downloaded_this_month == 0:
            print("No AM or PM files this month.")
        else:
            print(
                f"Selected {selected_pass} for {month_tag}, "
                f"downloaded {downloaded_this_month} files."
            )

    print("\nDONE. Raw netCDF saved under:", OUT_ROOT / "raw")


if __name__ == "__main__":
    main()

AOI bounds EPSG:4326: (124.97675, -9.266, 125.93995, -8.3028)
AOI center: 125.45835 -8.7844

=== VIIRS PASS SEARCH ORDER ===
SCENE_CENTER_TIME UTC: 01:46:54.3099020Z
UTC hour: 1.782
Approx local solar hour: 10.146
Search order: ['AM', 'PM']
Preferred short name: L3S_LEO_AM-STAR-v2.81
Fallback short name: L3S_LEO_PM-STAR-v2.81
Authenticated: True

=== DOWNLOAD SETTINGS ===
Date range: 2020-01-01 to 2025-12-31
Output root: ../Timor_part1/L3S_STAR
Search order: ['AM', 'PM']

=== Download month 2020-01 ===
[SEARCH] L3S_LEO_AM-STAR-v2.81 2020-01-01..2020-01-31 -> 0 granules
No AM files this month. Trying next pass.
[SEARCH] L3S_LEO_PM-STAR-v2.81 2020-01-01..2020-01-31 -> 62 granules


/home/mingyue/apps/miniforge3/envs/sst/lib/python3.12/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/home/mingyue/apps/miniforge3/envs/sst/lib/python3.12/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/62 [00:00<?, ?it/s]

...

[DL] -> 60 files
Selected PM for 2025-11, downloaded 60 files.

=== Download month 2025-12 ===
[SEARCH] L3S_LEO_AM-STAR-v2.81 2025-12-01..2025-12-31 -> 0 granules
No AM files this month. Trying next pass.
[SEARCH] L3S_LEO_PM-STAR-v2.81 2025-12-01..2025-12-31 -> 62 granules


QUEUEING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/62 [00:00<?, ?it/s]

[DL] -> 62 files
Selected PM for 2025-12, downloaded 62 files.

DONE. Raw netCDF saved under: ../Timor_part1/L3S_STAR/raw


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta
import earthaccess
from shapely.geometry import box

# =========================
# USER CONFIG
# =========================
START_DATE = "2020-01-01"
END_DATE   = "2025-12-31"

# Landsat scene UTC center time
# Used only to decide preferred VIIRS AM/PM search order
SCENE_CENTER_TIME = "01:40:44.6655680Z"

# =========================
# AOI: lon, lat in EPSG:4326
# =========================
BUFFER_DEG = 0.05

points = [
    (126.3977, -8.2006),
    (126.9965, -8.2387),
    (127.0624, -9.0858),
    (126.3043, -9.1129),
]

lons = [p[0] for p in points]
lats = [p[1] for p in points]

min_lon, max_lon = min(lons), max(lons)
min_lat, max_lat = min(lats), max(lats)

width = max_lon - min_lon
height = max_lat - min_lat
side = max(width, height) + 2 * BUFFER_DEG

center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2
half_side = side / 2

square_min_lon = center_lon - half_side
square_max_lon = center_lon + half_side
square_min_lat = center_lat - half_side
square_max_lat = center_lat + half_side

square_geom_4326 = box(
    square_min_lon,
    square_min_lat,
    square_max_lon,
    square_max_lat,
)

bbox_search = square_geom_4326.bounds  # west, south, east, north

print("AOI bounds EPSG:4326:", bbox_search)
print("AOI center:", center_lon, center_lat)

# =========================
# VIIRS CONFIG
# =========================
SHORT_NAME_AM = "L3S_LEO_AM-STAR-v2.81"
SHORT_NAME_PM = "L3S_LEO_PM-STAR-v2.81"

OUT_ROOT = Path("../Timor_part2/L3S_STAR")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# AM / PM PREFERENCE
# =========================
def choose_preferred_viirs_pass(scene_center_time: str, center_lon: float):
    """
    Decide preferred VIIRS pass using Landsat scene center time.

    SCENE_CENTER_TIME is UTC.
    Approx local solar time = UTC hour + lon / 15.

    local solar time < 12  -> prefer AM first
    local solar time >= 12 -> prefer PM first

    If preferred pass has no granules, script will try the other pass.
    """
    time_str = scene_center_time.replace("Z", "")

    hh, mm, ss = time_str.split(":")
    utc_hour = int(hh) + int(mm) / 60.0 + float(ss) / 3600.0

    local_solar_hour = (utc_hour + center_lon / 15.0) % 24.0

    if local_solar_hour < 12:
        preferred = "AM"
        fallback = "PM"
    else:
        preferred = "PM"
        fallback = "AM"

    return preferred, fallback, utc_hour, local_solar_hour


PREFERRED_PASS, FALLBACK_PASS, UTC_HOUR, LOCAL_SOLAR_HOUR = choose_preferred_viirs_pass(
    SCENE_CENTER_TIME,
    center_lon,
)

SHORT_NAME_LOOKUP = {
    "AM": SHORT_NAME_AM,
    "PM": SHORT_NAME_PM,
}

SEARCH_ORDER = [PREFERRED_PASS, FALLBACK_PASS]

print("\n=== VIIRS PASS SEARCH ORDER ===")
print("SCENE_CENTER_TIME UTC:", SCENE_CENTER_TIME)
print("UTC hour:", round(UTC_HOUR, 3))
print("Approx local solar hour:", round(LOCAL_SOLAR_HOUR, 3))
print("Search order:", SEARCH_ORDER)
print("Preferred short name:", SHORT_NAME_LOOKUP[PREFERRED_PASS])
print("Fallback short name:", SHORT_NAME_LOOKUP[FALLBACK_PASS])

# =========================
# HELPERS
# =========================
def month_start(dt: datetime) -> datetime:
    return dt.replace(day=1)


def month_end(dt: datetime) -> datetime:
    if dt.month == 12:
        nxt = dt.replace(year=dt.year + 1, month=1, day=1)
    else:
        nxt = dt.replace(month=dt.month + 1, day=1)

    return nxt - timedelta(days=1)


def iter_months(start: datetime, end: datetime):
    cur = month_start(start)

    while cur <= end:
        me = month_end(cur)
        yield cur, min(me, end)

        if cur.month == 12:
            cur = cur.replace(year=cur.year + 1, month=1, day=1)
        else:
            cur = cur.replace(month=cur.month + 1, day=1)


def search_one_month(
    short_name: str,
    t0: datetime,
    t1: datetime,
    bbox4,
):
    results = earthaccess.search_data(
        short_name=short_name,
        provider="POCLOUD",
        temporal=(
            t0.strftime("%Y-%m-%dT00:00:00Z"),
            t1.strftime("%Y-%m-%dT23:59:59Z"),
        ),
        bounding_box=bbox4,
    )

    print(
        f"[SEARCH] {short_name} "
        f"{t0:%Y-%m-%d}..{t1:%Y-%m-%d} -> {len(results)} granules"
    )

    return results


def download_results(
    results,
    out_dir: Path,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(results) == 0:
        return []

    paths = earthaccess.download(
        results,
        local_path=str(out_dir),
    )

    paths = paths or []

    print(f"[DL] -> {len(paths)} files")
    return paths


# =========================
# MAIN
# =========================
def main():
    auth = earthaccess.login(
        strategy="interactive",
        persist=True,
    )

    print("Authenticated:", auth.authenticated)

    start = datetime.fromisoformat(START_DATE)
    end = datetime.fromisoformat(END_DATE)

    print("\n=== DOWNLOAD SETTINGS ===")
    print("Date range:", START_DATE, "to", END_DATE)
    print("Output root:", OUT_ROOT)
    print("Search order:", SEARCH_ORDER)

    for m0, m1 in iter_months(start, end):
        month_tag = m0.strftime("%Y-%m")

        print(f"\n=== Download month {month_tag} ===")

        downloaded_this_month = 0
        selected_pass = None

        for pass_name in SEARCH_ORDER:
            short_name = SHORT_NAME_LOOKUP[pass_name]

            results = search_one_month(
                short_name,
                m0,
                m1,
                bbox_search,
            )

            if len(results) == 0:
                print(f"No {pass_name} files this month. Trying next pass.")
                continue

            month_dir = OUT_ROOT / "raw" / month_tag / pass_name

            paths = download_results(
                results,
                month_dir,
            )

            downloaded_this_month += len(paths)
            selected_pass = pass_name

            # stop after first available pass
            break

        if downloaded_this_month == 0:
            print("No AM or PM files this month.")
        else:
            print(
                f"Selected {selected_pass} for {month_tag}, "
                f"downloaded {downloaded_this_month} files."
            )

    print("\nDONE. Raw netCDF saved under:", OUT_ROOT / "raw")


if __name__ == "__main__":
    main()

AOI bounds EPSG:4326: (126.17719999999998, -9.162899999999999, 127.1895, -8.150599999999999)
AOI center: 126.68334999999999 -8.656749999999999

=== VIIRS PASS SEARCH ORDER ===
SCENE_CENTER_TIME UTC: 01:40:44.6655680Z
UTC hour: 1.679
Approx local solar hour: 10.125
Search order: ['AM', 'PM']
Preferred short name: L3S_LEO_AM-STAR-v2.81
Fallback short name: L3S_LEO_PM-STAR-v2.81
Authenticated: True

=== DOWNLOAD SETTINGS ===
Date range: 2020-01-01 to 2025-12-31
Output root: ../Timor_part2/L3S_STAR
Search order: ['AM', 'PM']

=== Download month 2020-01 ===
[SEARCH] L3S_LEO_AM-STAR-v2.81 2020-01-01..2020-01-31 -> 0 granules
No AM files this month. Trying next pass.
[SEARCH] L3S_LEO_PM-STAR-v2.81 2020-01-01..2020-01-31 -> 62 granules


QUEUEING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/62 [00:00<?, ?it/s]

[DL] -> 62 files
Selected PM for 2020-01, downloaded 62 files.

=== Download month 2020-02 ===
[SEARCH] L3S_LEO_AM-STAR-v2.81 2020-02-01..2020-02-29 -> 0 granules
No AM files this month. Trying next pass.
[SEARCH] L3S_LEO_PM-STAR-v2.81 2020-02-01..2020-02-29 -> 58 granules


...

[DL] -> 60 files
Selected PM for 2025-11, downloaded 60 files.

=== Download month 2025-12 ===
[SEARCH] L3S_LEO_AM-STAR-v2.81 2025-12-01..2025-12-31 -> 0 granules
No AM files this month. Trying next pass.
[SEARCH] L3S_LEO_PM-STAR-v2.81 2025-12-01..2025-12-31 -> 62 granules


QUEUEING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/62 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/62 [00:00<?, ?it/s]

[DL] -> 62 files
Selected PM for 2025-12, downloaded 62 files.

DONE. Raw netCDF saved under: ../Timor_part2/L3S_STAR/raw
